# Collections and views: analyses over many records

A record stores what a build produced. Analyses derive more from it: a model object rebuilt
from stored weights, responses to new stimuli, summaries across an ensemble. This notebook
uses

- `td.View`, a wrapper around one record for derived objects, with `td.cached_method` (an
  in-memory cache per view) and `view.memory` (a joblib cache inside the record), and
- `Collection.map`, which applies a function to every record of a collection and stacks the
  xarray results along a new dimension labeled by spec fields,

as flyvis's `NetworkView` and `EnsembleView` do for trained networks.

In [ ]:
import functools
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr
from jaxtyping import Float

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-views-"))
td.set_root(root)

## An ensemble of fitted models

`Tuning` fits a linear-nonlinear model to noisy responses: one weight vector over a bank of
Gaussian basis functions per record, with checkpoints along the fit.

In [ ]:
class Tuning(td.Record):
    __record_name__ = "Tuning"

    class Spec(td.Spec):
        n_basis: int = 8
        noise: float = 0.2
        seed: int = 0
        n_steps: int = 300
        checkpoint_every: int = 100

    weights: Float[np.ndarray, "n_checkpoints n_basis"]

    def build(self, spec):
        rng = np.random.default_rng(spec.seed)
        x = np.linspace(-np.pi, np.pi, 64)
        target = np.exp(np.cos(x - 0.5)) + spec.noise * rng.normal(size=x.size)
        features = basis(x, spec.n_basis)
        w = np.zeros(spec.n_basis)
        for step in range(1, spec.n_steps + 1):
            w -= 0.05 * features.T @ (features @ w - target) / len(x)
            if step % spec.checkpoint_every == 0:
                self.append("weights", w[None])
                self.append("checkpoint_step", [step])


def basis(x, n_basis):
    """Gaussian bumps on the circle: (len(x), n_basis)."""
    centres = np.linspace(-np.pi, np.pi, n_basis, endpoint=False)
    return np.exp(-(np.angle(np.exp(1j * (x[:, None] - centres))) ** 2) / 0.5)


ensemble = td.Collection(Tuning.get({"seed": seed, "noise": noise}) for noise in (0.1, 0.5) for seed in range(3))
ensemble

## A view rebuilds domain objects

A `View` holds its record and nothing else; unknown attributes resolve on the record
(`view.spec`, `view.path`). Methods decorated with `td.cached_method` are cached per view
instance, keyed by the canonical form of their arguments (arrays by digest, records by key).

In [ ]:
class Model:
    """The fitted tuning curve at one checkpoint."""

    def __init__(self, weights, n_basis):
        self.weights, self.n_basis = weights, n_basis

    def __call__(self, x):
        return basis(np.asarray(x), self.n_basis) @ self.weights


def responses(view, x, step=None, batch_size=16):
    """Model responses to orientations `x` as a labeled array (computed in batches)."""
    model = view.model(step)
    y = np.concatenate([model(batch) for batch in np.array_split(x, max(1, len(x) // batch_size))])
    return xr.DataArray(y, dims="stimulus", coords={"orientation": ("stimulus", x)}, name="response")


class TuningView(td.View):
    @functools.cached_property
    def memory(self):
        """The record's joblib cache, without joblib's progress messages."""
        return td.memory(self.record.path / "__cache__", verbose=0)

    @td.cached_method(maxsize=4)
    def model(self, step: int | None = None) -> Model:
        steps = list(self.record.load("checkpoint_step"))
        index = len(steps) - 1 if step is None else steps.index(step)
        return Model(self.record["weights"][index], self.spec.n_basis)

    def responses(self, x: np.ndarray, step: int | None = None, batch_size: int = 16) -> xr.DataArray:
        # cached on disk in <record>/__cache__; batch_size does not change the result
        return self.memory.cache(responses, ignore=["batch_size"])(self, x, step, batch_size)


view = TuningView(ensemble[0])
print(view, view.spec.seed, view.model() is view.model(), view.model(100) is view.model())

`view.memory` is a joblib `Memory` in the record's `__cache__` directory (by default
`record.memory`; this view overrides it only to silence joblib). A view argument
hashes by its record's identity (type and key), so joblib never hashes stored weights.
`view.needs_compute(cached_function, *args)` tells whether a call would compute or load.

In [ ]:
x = np.linspace(-np.pi, np.pi, 33)
cached = view.memory.cache(responses, ignore=["batch_size"])
print("needs compute:", view.needs_compute(cached, view, x, None, 16))
first = view.responses(x)
print("needs compute:", view.needs_compute(cached, view, x, None, 4), "(batch_size is ignored)")
print("same values:", bool((view.responses(x, batch_size=4) == first).all()))
view.record.tree()

Results from the cache come back through tensordir's codecs: an xarray result is stored as
zarr and reopened lazily, so its coordinates survive.

In [ ]:
view.responses(x).sel(stimulus=slice(0, 4))

`td.needs_compute(func, *args)` is the same check for any memoized function, and
`cached_method.cache_clear(view)` empties the in-memory cache of one view.

In [ ]:
print(td.needs_compute(cached, view, x + 1, None, 16))
TuningView.model.cache_clear(view)
print(view.model() is not None)  # rebuilt on the next call

## Map over a collection

`Collection.map(fn, dim=, coords=)` calls `fn(record)` for every record and concatenates the
xarray results along `dim`, indexed by record key, with the named spec fields as coordinates
along `dim`. Pass `memory=` (a joblib `Memory`) to memoize `fn` per record.

In [ ]:
def final_responses(record):
    return TuningView(record).responses(x)


stacked = ensemble.map(final_responses, dim="model", coords=["seed", "noise"])
stacked

The spec coordinates make the stacked result easy to group and filter.

In [ ]:
print(stacked.groupby("noise").std("model").mean("stimulus").round(4).to_series())
td.where(stacked, noise=0.5, seed=">0").coords

Collections come from queries as well as from iterables. With `memory=`, each result is
cached, so a second `map` over the same records loads instead of computing.

In [ ]:
memory = td.memory(root / "cache", verbose=0)


def peak(record):
    curve = final_responses(record)
    return xr.Dataset({"peak": curve.max(), "preferred": curve.orientation[curve.argmax("stimulus")]})


peaks = Tuning.query(noise=0.1).map(peak, dim="model", coords=["seed"], memory=memory)
print("would compute again:", td.needs_compute(memory.cache(peak), ensemble[0]))
peaks

## Views pickle as their record

A view pickles as its record's identity `(type, key, path)`, so views can be passed to worker
processes and joblib caches cheaply; the record is reopened on the other side.

In [ ]:
import pickle

payload = pickle.dumps(view)
restored = pickle.loads(payload)
print(len(payload), "bytes;", restored.record == view.record, restored.model().weights.round(2))